In [3]:
import sys
print(sys.executable)


c:\Users\18721\miniconda3\envs\tf_gpu\python.exe


In [9]:
import tensorflow as tf
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))




TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [7]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

from tensorflow.python.platform import build_info
print("CUDA version:", build_info.build_info.get('cuda_version', 'N/A'))
print("cuDNN version:", build_info.build_info.get('cudnn_version', 'N/A'))




TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [8]:
# ==========================================================
# INbreast V-UNet (Feature Extraction Transfer Learning)
# Encoder: VGG16 pretrained on ImageNet (Frozen)
# Decoder: Trainable U-Net decoder for segmentation
# Output size: 512x512x1 (matches masks)
# ==========================================================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
tf.get_logger().setLevel("ERROR")


# 1) 看看 TensorFlow 看到没看到 GPU
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Physical GPUs:", tf.config.list_physical_devices('GPU'))

# 2) 避免一次性吃满显存（很多笔记本/台机默认会 OOM）
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print("set_memory_growth failed:", e)

# 3) 混合精度（NVIDIA RTX/GTX/50 系列推荐，速度通常明显提升）
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print("Compute dtype policy:", mixed_precision.global_policy())


# ----------------------------------------------------------
# 1️⃣ 数据加载
# ----------------------------------------------------------
base_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_Unet")
train_img_dir = base_dir / "train" / "images"
train_mask_dir = base_dir / "train" / "labels"
val_img_dir   = base_dir / "val" / "images"
val_mask_dir  = base_dir / "val" / "labels"
test_img_dir  = base_dir / "test" / "images"
test_mask_dir = base_dir / "test" / "labels"

IMG_SIZE = (512, 512)
BATCH_SIZE = 4
SEED = 2025

def load_data(img_dir, mask_dir, img_size=IMG_SIZE):
    imgs, masks = [], []
    img_files = sorted(list(img_dir.glob("*")))
    for img_path in img_files:
        # 依据文件名前缀匹配标签（你原始逻辑）
        prefix = img_path.stem[:8]
        mask_candidates = list(mask_dir.glob(f"{prefix}*"))
        if not mask_candidates:
            continue
        mask_path = mask_candidates[0]

        # 读取灰度图
        img  = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            continue

        img  = cv2.resize(img, img_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, img_size, interpolation=cv2.INTER_NEAREST)

        img  = img.astype(np.float32) / 255.0
        mask = (mask > 127).astype(np.float32)  # 二值化

        imgs.append(np.expand_dims(img, axis=-1))   # (H, W, 1)
        masks.append(np.expand_dims(mask, axis=-1)) # (H, W, 1)

    return np.array(imgs), np.array(masks)

X_train, y_train = load_data(train_img_dir, train_mask_dir)
X_val,   y_val   = load_data(val_img_dir, val_mask_dir)
X_test,  y_test  = load_data(test_img_dir, test_mask_dir)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ----------------------------------------------------------
# 2️⃣ 数据增强（轻度）
# 说明：ImageDataGenerator 在 flow(X, y) 模式下会对 X/Y 应用相同的几何变换。
# 对 mask 我们已用最近邻在预处理 resize；增广阶段的小角度/平移影响很小。
# 若需严格最近邻增广，可改用 tf.image / Albumentations 自定义。
# ----------------------------------------------------------
train_idg = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_idg = ImageDataGenerator()

train_gen = train_idg.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED, shuffle=True)
val_gen   = val_idg.flow(X_val, y_val, batch_size=BATCH_SIZE, seed=SEED, shuffle=False)

# ----------------------------------------------------------
# 3️⃣ 评价指标（Dice / IoU）
# ----------------------------------------------------------
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - inter
    return (inter + smooth) / (union + smooth)

# ----------------------------------------------------------
# 4️⃣ 构建 V-UNet（冻结 VGG16 编码器）
# 关键修复：
#   - s1 = block1_conv2（尺寸 512x512）
#   - 解码器最后再上采样一次 256→512，并与 s1 融合
#   - 伪 RGB 后使用 VGG16 的 preprocess_input
# ----------------------------------------------------------
def build_vunet_frozen(input_shape=(512, 512, 1), lr=1e-4):
    # VGG16 backbone（预训练 + 冻结）
    vgg = applications.VGG16(include_top=False, weights='imagenet', input_shape=(512, 512, 3))
    for layer in vgg.layers:
        layer.trainable = False

    # 输入：灰度 -> 伪 RGB
    inputs = layers.Input(shape=input_shape)
    x_rgb = layers.Concatenate(axis=-1)([inputs, inputs, inputs])  # (H, W, 3)

    # 预处理到 VGG16 习惯的分布（BGR/mean-substract 在 preprocess 内处理）
    x_rgb255 = layers.Lambda(lambda z: z * 255.0)(x_rgb)
    x_pp = layers.Lambda(vgg16_preprocess)(x_rgb255)

    # 取各层输出（注意 s1 用未池化的 block1_conv2 → 512×512）
    s1 = vgg.get_layer("block1_conv2").output  # 512x512x64
    s2 = vgg.get_layer("block2_pool").output   # 128x128x128
    s3 = vgg.get_layer("block3_pool").output   # 64x64x256
    s4 = vgg.get_layer("block4_pool").output   # 32x32x512
    b  = vgg.get_layer("block5_pool").output   # 16x16x512

    # 定义 encoder 模型，并在 x_pp 上执行
    encoder = models.Model(inputs=vgg.input, outputs=[s1, s2, s3, s4, b], name="vgg16_encoder")
    s1, s2, s3, s4, b = encoder(x_pp)

    # Decoder
    # 16→32
    x = layers.Conv2DTranspose(512, (2,2), strides=(2,2), padding='same')(b)
    x = layers.Concatenate()([x, s4])
    x = layers.Conv2D(512, (3,3), activation='relu', padding='same')(x)
    x = layers.Conv2D(512, (3,3), activation='relu', padding='same')(x)

    # 32→64
    x = layers.Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(x)
    x = layers.Concatenate()([x, s3])
    x = layers.Conv2D(256, (3,3), activation='relu', padding='same')(x)
    x = layers.Conv2D(256, (3,3), activation='relu', padding='same')(x)

    # 64→128
    x = layers.Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(x)
    x = layers.Concatenate()([x, s2])
    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)

    # 128→256（注意：这里没有 skip，因为 s1 是 512 尺寸）
    x = layers.Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(x)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)

    # 256→512（最后一次上采样 + 与 s1 融合）
    x = layers.Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(x)
    x = layers.Concatenate()([x, s1])
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)

    outputs = layers.Conv2D(1, (1,1), activation='sigmoid', dtype='float32')(x)

    model = models.Model(inputs, outputs, name="VUNet_VGG16_Frozen")
    model.compile(
        optimizer=optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy', dice_coef, iou_coef]
    )
    return model

model = build_vunet_frozen()
model.summary()

# ----------------------------------------------------------
# 5️⃣ 训练
# ----------------------------------------------------------
EPOCHS = 30

callbacks = [
    EarlyStopping(monitor='val_dice_coef', mode='max', patience=8, restore_best_weights=True),
    ModelCheckpoint('vunet_best.h5', monitor='val_dice_coef', mode='max', save_best_only=True, verbose=1)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    steps_per_epoch=len(train_gen),
    validation_steps=len(val_gen),
    callbacks=callbacks,
    verbose=1
)




TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [3]:
# ----------------------------------------------------------
# 6️⃣ 评估与可视化
# ----------------------------------------------------------
loss, acc, dice, iou = model.evaluate(X_test, y_test, verbose=0)
print(f"Test  Loss: {loss:.4f}\nTest  Acc : {acc:.4f}\nTest  Dice: {dice:.4f}\nTest  IoU : {iou:.4f}")

# 预测与可视化
pred = model.predict(X_test[:3])
pred_bin = (pred > 0.9).astype(np.float32)

for i in range(min(3, len(X_test))):
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    plt.imshow(X_test[i].squeeze(), cmap='gray')
    plt.title("Original"); plt.axis('off')

    plt.subplot(1,3,2)
    plt.imshow(y_test[i].squeeze(), cmap='gray')
    plt.title("Ground Truth"); plt.axis('off')

    plt.subplot(1,3,3)
    plt.imshow(pred_bin[i].squeeze(), cmap='gray')
    plt.title("Prediction (thr=0.5)"); plt.axis('off')
    plt.tight_layout()
    plt.show()


NameError: name 'model' is not defined

In [ ]:
# ==========================================================
# ResUNet for INbreast-style dataset
# Encoder: ResNet50 (ImageNet)  -- frozen -> fine-tune
# Decoder: UNet upsampling path
# I/O: gray 512x512 (-> pseudo RGB)  --> 512x512x1 mask
# ==========================================================

import os, cv2, numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from pathlib import Path
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications.resnet import ResNet50, preprocess_input as resnet_preprocess

tf.get_logger().setLevel("ERROR")

# ---- GPU: 按需分配（DirectML/CUDA 都安全） ----
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except: pass

# 可开可关：混合精度
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
print("Policy:", mixed_precision.global_policy())
print("GPUs:", tf.config.list_physical_devices('GPU'))

# ---- Paths & Params ----
base_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_Unet")
train_img_dir = base_dir / "train" / "images"
train_mask_dir = base_dir / "train" / "labels"
val_img_dir   = base_dir / "val"   / "images"
val_mask_dir  = base_dir / "val"   / "labels"
test_img_dir  = base_dir / "test"  / "images"
test_mask_dir = base_dir / "test"  / "labels"

IMG_SIZE = (512, 512)
BATCH    = 4
SEED     = 2025
AUTOTUNE = tf.data.AUTOTUNE

# ---- 读取为 numpy：灰度→伪RGB，mask 二值 ----
def crop_to_breast(img, mask=None, margin=10):
    _, binm = cv2.threshold(img, 0, 255, cv2.THRESH_OTSU)
    cnts,_ = cv2.findContours((binm>0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return img, mask
    x,y,w,h = cv2.boundingRect(max(cnts, key=cv2.contourArea))
    x = max(0,x-margin); y = max(0,y-margin)
    xe = min(img.shape[1], x+w+margin); ye = min(img.shape[0], y+h+margin)
    img = img[y:ye, x:xe]
    if mask is not None: mask = mask[y:ye, x:xe]
    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_LINEAR)
    if mask is not None: mask = cv2.resize(mask, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    return img, mask

def load_pairs(img_dir: Path, mask_dir: Path):
    X, Y = [], []
    for p in sorted(img_dir.glob("*")):
        prefix = p.stem[:8]
        mlist = list(mask_dir.glob(f"{prefix}*"))
        if not mlist: continue
        m = mlist[0]
        img  = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(m), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None: continue
        img, mask = crop_to_breast(img, mask)  # 可注释掉试试
        img3 = np.stack([img, img, img], axis=-1).astype(np.float32)   # 0~255
        mask = (mask > 127).astype(np.float32)[..., None]
        X.append(img3); Y.append(mask)
    return np.array(X), np.array(Y)

X_train, y_train = load_pairs(train_img_dir, train_mask_dir)
X_val,   y_val   = load_pairs(val_img_dir,   val_mask_dir)
X_test,  y_test  = load_pairs(test_img_dir,  test_mask_dir)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# ---- tf.data + tf.image（mask 最近邻）----
def tf_augment(x3, y):
    # x3: 0~255
    # 轻度几何增强
    do_lr = tf.random.uniform([]) > 0.5
    x3 = tf.cond(do_lr, lambda: tf.image.flip_left_right(x3), lambda: x3)
    y  = tf.cond(do_lr, lambda: tf.image.flip_left_right(y),  lambda: y)

    do_ud = tf.random.uniform([]) > 0.85
    x3 = tf.cond(do_ud, lambda: tf.image.flip_up_down(x3), lambda: x3)
    y  = tf.cond(do_ud, lambda: tf.image.flip_up_down(y),  lambda: y)

    zoom = tf.random.uniform([], 0.9, 1.05)
    new_hw = tf.cast(tf.round(zoom*tf.cast(tf.shape(x3)[:2], tf.float32)), tf.int32)
    x3 = tf.image.resize(x3, new_hw, method='bilinear')
    y  = tf.image.resize(y,  new_hw, method='nearest')
    x3 = tf.image.resize_with_crop_or_pad(x3, IMG_SIZE[0], IMG_SIZE[1])
    y  = tf.image.resize_with_crop_or_pad(y,  IMG_SIZE[0], IMG_SIZE[1])
    # ResNet 预处理（-123.68 等均值，BGR 排序在内部完成）
    x3 = resnet_preprocess(x3)     # 期望 0~255
    return x3, y

def tf_prep_val(x3, y):
    return resnet_preprocess(x3), y

def make_ds(X, Y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if training:
        ds = ds.shuffle(1024, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(tf_augment, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(tf_prep_val, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val,   y_val,   training=False)
test_ds  = make_ds(X_test,  y_test,  training=False)

# ---- 指标 & 损失（Focal-Tversky）----
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1]); y_pred_f = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.*inter + smooth) / (tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)

def iou_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1]); y_pred_f = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)-inter
    return (inter+smooth)/(union+smooth)

def tversky(y_true, y_pred, alpha=0.7, beta=0.3, smooth=1e-6):
    yt = tf.reshape(y_true, [-1]); yp = tf.reshape(y_pred, [-1])
    tp = tf.reduce_sum(yt*yp)
    fp = tf.reduce_sum((1-yt)*yp)
    fn = tf.reduce_sum(yt*(1-yp))
    return (tp+smooth)/(tp+alpha*fp+beta*fn+smooth)

def focal_tversky_loss(y_true, y_pred, gamma=0.75):
    return tf.pow(1.0 - tversky(y_true, y_pred), gamma)

# ---- ResUNet 架构 ----
def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.Activation('relu')(x)
    x = layers.Conv2D(f, 3, padding='same')(x); x = layers.Activation('relu')(x)
    return x

def resunet(input_shape=(512,512,3), lr=1e-4, freeze_encoder=True):
    # Encoder: ResNet50 (include_top=False)
    backbone = ResNet50(include_top=False, weights='imagenet', input_shape=input_shape)
    if freeze_encoder:
        for l in backbone.layers: l.trainable = False

    # 取 skip
    s1 = backbone.get_layer('conv1_relu').output      # 256x256 x64
    s2 = backbone.get_layer('conv2_block3_out').output# 128x128 x256
    s3 = backbone.get_layer('conv3_block4_out').output# 64x64  x512
    s4 = backbone.get_layer('conv4_block6_out').output# 32x32  x1024
    b  = backbone.get_layer('conv5_block3_out').output# 16x16  x2048

    inputs = backbone.input
    # Decoder
    x = layers.Conv2DTranspose(1024, 2, strides=2, padding='same')(b) # 16->32
    x = layers.Concatenate()([x, s4]); x = conv_block(x, 512)

    x = layers.Conv2DTranspose(512, 2, strides=2, padding='same')(x)  # 32->64
    x = layers.Concatenate()([x, s3]); x = conv_block(x, 256)

    x = layers.Conv2DTranspose(256, 2, strides=2, padding='same')(x)  # 64->128
    x = layers.Concatenate()([x, s2]); x = conv_block(x, 128)

    x = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(x)  # 128->256
    x = layers.Concatenate()([x, s1]); x = conv_block(x, 64)

    x = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(x)   # 256->512
    x = conv_block(x, 32)

    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32')(x)
    model = models.Model(inputs, outputs, name='ResUNet50')
    model.compile(optimizer=optimizers.Adam(lr),
                  loss=focal_tversky_loss,
                  metrics=['accuracy', dice_coef, iou_coef])
    return model

model = resunet()
model.summary()

# ---- 训练：两阶段 ----
cbs = [
    EarlyStopping(monitor='val_dice_coef', mode='max', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_dice_coef', mode='max', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint('resunet_best.h5', monitor='val_dice_coef', mode='max', save_best_only=True, verbose=1)
]

# 阶段1：冻结编码器
history1 = model.fit(train_ds, validation_data=val_ds, epochs=20, verbose=1, callbacks=cbs)

# 阶段2：解冻 conv4/conv5 微调
for l in model.layers:
    if hasattr(l, "name") and (l.name.startswith("conv5_") or l.name.startswith("conv4_")):
        l.trainable = True

model.compile(optimizer=optimizers.Adam(1e-5),
              loss=focal_tversky_loss,
              metrics=['accuracy', dice_coef, iou_coef])

history2 = model.fit(train_ds, validation_data=val_ds, epochs=30, verbose=1, callbacks=cbs)






ModuleNotFoundError: No module named 'cv2'

In [9]:
# ---- 评估 ----
loss, acc, dice, iou = model.evaluate(test_ds, verbose=0)
print(f"Test  Loss: {loss:.4f}\nTest  Acc : {acc:.4f}\nTest  Dice: {dice:.4f}\nTest  IoU : {iou:.4f}")

# ---- 可视化 ----
def show_preds_np(X3, Y, model, k=3, thr=0.25):
    # 需与训练同样预处理
    Xp = resnet_preprocess(X3[:k].copy())
    P  = model.predict(Xp, verbose=0)
    Pb = (P>thr).astype(np.float32)
    for i in range(min(k, len(X3))):
        plt.figure(figsize=(12,4))
        plt.subplot(1,3,1); plt.imshow(X3[i][...,0].astype(np.uint8), cmap='gray'); plt.title('Original'); plt.axis('off')
        plt.subplot(1,3,2); plt.imshow(Y[i].squeeze(), cmap='gray'); plt.title('Ground Truth'); plt.axis('off')
        plt.subplot(1,3,3); plt.imshow(Pb[i].squeeze(), cmap='gray'); plt.title(f'Prediction (thr={thr})'); plt.axis('off')
        plt.tight_layout(); plt.show()

show_preds_np(X_test, y_test, model, k=3, thr=0.5)

NameError: name 'model' is not defined